In [ ]:
%%configure -f
{
  "driverMemory": "28g",
  "executorMemory": "28g",
  "conf": {
    "spark.microsoft.delta.optimizeWrite.enabled": "true",
    "spark.databricks.delta.optimizeWrite.binSize": "1073741824",
    "spark.databricks.delta.autoCompact.enabled": "true",
    "spark.native.enabled": "true"
  }
}

# Bronze Layer - Raw Data Ingestion

This notebook ingests raw financial invoice data from generated CSV/JSON files into the Bronze layer Delta tables.

**Source Data:**
- `data/sample/cold_path_invoices.csv` - Historical batch invoices (500 records)
- `data/sample/hot_stream_batch.json` - Real-time streaming events

**Target:** Bronze Delta tables with raw, untransformed data plus ingestion metadata.

In [ ]:
# Import libraries
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType, DateType
import datetime

# Get runtime context
context = notebookutils.runtime.context
isForPipeline = context["isForPipeline"]
workspace_id = context["currentWorkspaceId"]
default_lakehouse_id = context.get("defaultLakehouseId")
default_lakehouse_name = context.get("defaultLakehouseName")

print(f"isForPipeline: {isForPipeline}")
print(f"workspace_id: {workspace_id}")
print(f"default_lakehouse_id: {default_lakehouse_id}")
print(f"default_lakehouse_name: {default_lakehouse_name}")

In [ ]:
# Define explicit schema for invoice data (avoid inferSchema overhead)
invoice_schema = StructType([
    StructField("invoice_id", StringType(), False),
    StructField("timestamp", StringType(), False),  # Keep as string initially, parse later
    StructField("date", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("region", StringType(), False),
    StructField("product_category", StringType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("discount", DoubleType(), False),
    StructField("gross_amount", DoubleType(), False),
    StructField("net_revenue", DoubleType(), False),
    StructField("currency", StringType(), False),
    StructField("payment_status", StringType(), False)
])

print("Schema defined successfully")

In [ ]:
# Read cold path CSV data
cold_path_df = spark.read \
    .option("header", "true") \
    .schema(invoice_schema) \
    .csv("Files/cold_path_invoices.csv")

print(f"Cold path records: {cold_path_df.count()}")
cold_path_df.printSchema()
cold_path_df.show(5, truncate=False)

In [ ]:
# Read hot path JSON data
hot_path_df = spark.read \
    .option("multiline", "true") \
    .schema(invoice_schema) \
    .json("Files/hot_stream_batch.json")

print(f"Hot path records: {hot_path_df.count()}")
hot_path_df.printSchema()
hot_path_df.show(5, truncate=False)

In [ ]:
# Union both datasets and add ingestion metadata
from pyspark.sql import functions as F

# Add ingestion timestamp and source path
ingestion_timestamp = F.current_timestamp()

bronze_df = cold_path_df.unionByName(hot_path_df) \
    .withColumn("ingestion_timestamp", ingestion_timestamp) \
    .withColumn("source_system", F.lit("financial_generator")) \
    .withColumn("pipeline_run_id", F.lit(context.get("currentJobId", "manual_run")))

# Parse timestamp and date columns
bronze_df = bronze_df \
    .withColumn("event_timestamp", F.to_timestamp(F.col("timestamp"))) \
    .withColumn("event_date", F.to_date(F.col("date")))

print(f"Total bronze records: {bronze_df.count()}")
bronze_df.show(5, truncate=False)

In [ ]:
# Check if lakehouse is schema-enabled
def lakehouse_is_schema_enabled():
    import uuid
    probe = f"_schema_probe_{uuid.uuid4().hex}"
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {probe}")
        spark.sql(f"DROP SCHEMA IF EXISTS {probe}")
        return True
    except Exception as e:
        if "Feature not supported on Apache Spark in Microsoft Fabric" in str(e):
            return False
        raise

schema_enabled = lakehouse_is_schema_enabled()
print(f"Lakehouse schema-enabled: {schema_enabled}")

In [ ]:
# Write to Bronze layer
if schema_enabled:
    # Create bronze schema if not exists
    spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
    table_name = "bronze.raw_invoices"
else:
    table_name = "bronze_raw_invoices"

# Write as Delta table with overwrite mode (idempotent for batch reprocessing)
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✅ Bronze table written: {table_name}")

In [ ]:
# Verify Bronze table
bronze_table = spark.table(table_name)
print(f"Bronze record count: {bronze_table.count()}")
print(f"Bronze schema:")
bronze_table.printSchema()

# Show sample data
bronze_table.select("invoice_id", "customer_id", "region", "product_category", "net_revenue", "ingestion_timestamp").show(10, truncate=False)

In [ ]:
# Data quality checks on Bronze layer
from pyspark.sql import functions as F

total_records = bronze_table.count()
null_invoice_ids = bronze_table.filter(F.col("invoice_id").isNull()).count()
null_customer_ids = bronze_table.filter(F.col("customer_id").isNull()).count()
negative_revenue = bronze_table.filter(F.col("net_revenue") < 0).count()
duplicate_invoices = bronze_table.groupBy("invoice_id").count().filter(F.col("count") > 1).count()

print("=== BRONZE LAYER DATA QUALITY REPORT ===")
print(f"Total Records: {total_records}")
print(f"Null Invoice IDs: {null_invoice_ids}")
print(f"Null Customer IDs: {null_customer_ids}")
print(f"Negative Revenue Records: {negative_revenue}")
print(f"Duplicate Invoice IDs: {duplicate_invoices}")

# Record-level quality flag
bronze_with_quality = bronze_table.withColumn(
    "data_quality_flag",
    F.when(F.col("invoice_id").isNull(), "MISSING_INVOICE_ID")
    .when(F.col("customer_id").isNull(), "MISSING_CUSTOMER_ID")
    .when(F.col("net_revenue") < 0, "NEGATIVE_REVENUE")
    .otherwise("VALID")
)

quality_summary = bronze_with_quality.groupBy("data_quality_flag").count().orderBy(F.desc("count"))
quality_summary.show()

In [ ]:
# Enable Change Data Feed (CDF) for incremental Silver processing
spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"✅ CDF enabled on {table_name}")

# Show table properties
spark.sql(f"SHOW TBLPROPERTIES {table_name}").show(truncate=False)